# Notebook 24 — LiveMath-Judge as the sole scorer, all 300

A separate pilot. **Nothing here modifies `strict_v1`, `strict_v2` or the audit
diagnostics** — this is a third, independent scorer reported alongside them.

**Why.** The 2026-08-12 audit found that of the false passes a human
confirmed, **14 of 16 were `extract_final_answer` choosing a wrong or partial
span** and only 2 were SymPy collapse. A judge that reads the whole answer
against the whole ground truth skips the extractor entirely.

**The framing is the experiment.** FERMAT's ground truth is `pert_a` — the page
*with a deliberately injected error* on 150 of the 300 items. The task is
transcription fidelity, so a faithful copy of a wrong answer is CORRECT and a
silently repaired one is WRONG. `jnanliu/LiveMath-Judge` is trained for
*mathematical equivalence*, and its own criterion 2 says equivalent formulas
count as correct — the same acceptance that made Math-Verify unsafe here.

**So the gate decides whether this experiment is usable at all.** Items 55 and
273 are the two confirmed cases where the model silently repaired the injected
error. Under the fidelity prompt both must come back `no`. If either does not,
the judge is grading mathematics and the notebook stops before spending the
300-item run.

The native model-card prompt is run on the gate too, for comparison, but
**does not control anything** — it is recorded so the effect of the amendment
is measurable rather than assumed.

In [ ]:
# Auth + code access. GPU needed: LiveMath-Judge is a 3B Qwen2.5 fine-tune.
import json
import os
import sys

from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
RESULTS_DIR = f"{PROJECT_DIR}/results"
OUT_DIR = f"{PROJECT_DIR}/audit"
os.makedirs(OUT_DIR, exist_ok=True)

with open(f"{PROJECT_DIR}/.tokens.json") as f:
    HF_TOKEN = json.load(f)["HF_TOKEN"]
login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/
%pip install -q "antlr4-python3-runtime==4.11"

sys.path.insert(0, os.path.abspath("repo"))
for _name in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_name]
import importlib
importlib.invalidate_caches()

import pilot.audit_diagnostics
import pilot.canonicalize
import pilot.judge
import pilot.rescore
import pilot.strict_v2

print(f"pilot imported from: {os.path.dirname(pilot.judge.__file__)}")
assert pilot.canonicalize.latex_parser_available(), (
    "SymPy's LaTeX parser is NOT working -- strict_v1/v2 comparisons below "
    "would be computed against degraded labels.")
print("SymPy LaTeX parser OK")

import torch
assert torch.cuda.is_available(), "no GPU: enable a GPU runtime"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

RUN_CSV = "scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv"
run = pd.read_csv(f"{RESULTS_DIR}/{RUN_CSV}")
print(f"run: {len(run)} rows, model={run['model_id'].unique().tolist()}")
assert len(run) == 300

MODEL_ID = pilot.judge.MODEL_ID
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto", token=HF_TOKEN)
model.eval()
print(f"loaded {MODEL_ID}")


def make_backend(max_new_tokens=pilot.judge.MAX_NEW_TOKENS):
    """Wraps the model as a plain str -> str callable.

    Two corrections to the published usage snippet, neither of which fails
    loudly:
      * `apply_chat_template(..., return_tensors="pt")` returns a TENSOR, and
        the card then subscripts it as inputs["input_ids"]. Needs
        return_dict=True.
      * the card calls generate() with no max_new_tokens, so it defaults to
        20. The model writes an analysis BEFORE the boxed verdict, so 20
        tokens never reaches it and every item would parse as a failure that
        looks like the judge breaking rather than truncation.
    Only the NEWLY generated tokens are decoded, so the prompt's own literal
    \boxed{yes}/\boxed{no} can never be mistaken for the verdict.
    """
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    def backend(prompt: str) -> str:
        msgs = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(
            msgs, return_tensors="pt", return_dict=True,
            add_generation_prompt=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=pad_id)
        new = out[0][inputs["input_ids"].shape[1]:]
        return tokenizer.decode(new, skip_special_tokens=True)

    return backend


backend = make_backend()
print(f"backend ready (max_new_tokens={pilot.judge.MAX_NEW_TOKENS}, greedy)")

In [ ]:
# THE GATE. Runs before anything expensive and RAISES on failure.
#
# Items 55 and 273: the page reads `1 + tan x tan y` and `2/4`; the model
# wrote `1 -` and `3/4`. Both are silent repairs of FERMAT's injected error,
# so a fidelity judge must return `no`. Neither pair is mathematically
# equivalent, so even a pure equivalence judge should reject them -- a `yes`
# means the judge recalculated despite being told not to.
gate_fid = pilot.judge.run_gate(backend, run, fidelity=True)
gate_nat = pilot.judge.run_gate(backend, run, fidelity=False)

print("GATE — fidelity prompt (this one decides):")
for i, v in gate_fid["verdicts"].items():
    print(f"   item {i}: {v}   {'OK' if v == 'incorrect' else '<-- FAILS'}")
print(f"   passed: {gate_fid['passed']}")
print("\nGATE — native model-card prompt (recorded only, controls nothing):")
for i, v in gate_nat["verdicts"].items():
    print(f"   item {i}: {v}")
print(f"   would have passed: {gate_nat['passed']}")

print("\n--- raw judge output, fidelity prompt ---")
for i, raw in gate_fid["raw"].items():
    print(f"[item {i}] {str(raw)[:400]}\n")

if not gate_fid["passed"]:
    raise RuntimeError(
        "GATE FAILED: " + gate_fid["note"] +
        f" verdicts={gate_fid['verdicts']}. This is a REPORTABLE RESULT, not "
        "a bug -- record it as 'an open 3B math-equivalence judge cannot be "
        "repurposed as a fidelity judge on a perturbed-answer benchmark' and "
        "do not run the 300.")
print("GATE PASSED — proceeding to all 300.")

In [ ]:
# All 300, fidelity prompt only. ~20-40 min depending on GPU.
judged = pilot.judge.judge_run(backend, run, fidelity=True, progress=True)
print(f"judged {len(judged)} items")
print(judged["livemath_label"].value_counts(dropna=False).to_string())
print(f"parse failures: {int(judged['parse_failed'].sum())}")

judged_out = judged.copy()
judged_out.insert(0, "item_id", judged_out.index)
CSV_PATH = f"{OUT_DIR}/livemath_judge_all300_20260812.csv"
judged_out.to_csv(CSV_PATH, index=False)
print(f"\nper-item CSV -> {CSV_PATH}")
print(f"columns: {list(judged_out.columns)}")

In [ ]:
# Diagnostics + summary. strict_v1/strict_v2 are RECOMPUTED here for
# comparison only; neither rule is modified.
v1 = pilot.rescore.rescore_run(run, "strict_v1")["transcription_correct"].astype(bool)
v2 = pilot.strict_v2.rescore_v2(run)["correct_strict_v2_display_primary"].astype(bool)

audits = pilot.audit_diagnostics.load_audit_sets("repo/reference/audit")
dedup = {}
for name in pilot.audit_diagnostics.SET_PRECEDENCE:
    for i, r in audits[name].iterrows():
        dedup.setdefault(i, r["truth"])
human = pd.Series(dedup)
human = human[human != "indeterminate"]
print(f"determinate human labels available: {len(human)} "
      f"({int((human == 'correct').sum())} correct, {int((human == 'wrong').sum())} wrong)")

diag = pilot.judge.judge_diagnostics(judged, v1, v2, human=human)

# Example disagreements: judge vs both frozen rules, most informative first.
ex = judged.copy()
ex["strict_v1"] = v1.reindex(ex.index)
ex["strict_v2"] = v2.reindex(ex.index)
ex["item"] = ex.index
mask = (ex["verdict"] == "correct") != ex["strict_v1"]
examples = ex[mask].head(8)

MD_PATH = f"{OUT_DIR}/livemath_judge_all300_summary_20260812.md"
text = pilot.judge.write_summary_md(MD_PATH, diag, judged, gate_fid,
                                    native_gate=gate_nat, examples=examples)
print(f"summary -> {MD_PATH}\n")
print(text[:3000])
print("\n" + "=" * 70)
print("Download both files from Drive > uncertainty-math-vlm > audit/ and put")
print("them in the repo at reference/audit/ under the same names.")